# SimPT-Power quick start
A multi-voltage (0.4–400 kV) simulation database of mainland Portugal built from public data.
This notebook reads the open bundle (Parquet + JSON), which is an uncompressed subset of the frozen release, and shows what each layer contains.

| layer | what | files |
|---|---|---|
| HV core CORE-3787-REN | 60–400 kV network, 31,492 solved 15-min snapshots (May 2025 – Mar 2026) | `hv/` |
| MV | 617 root slices, PTD connections, inferred urban cables, MV customer estimate | `mv/` |
| LV | 72,434 secondary substations (PTD), geographic LV network LV-GEO-A | `lv/` |
| observations | public reference tables used for validation | `observations/` |

In [ ]:
import os, json
from pathlib import Path
def find_data():
    """Locate the SimPT-Power open bundle: $SIMPT_DATA, a Kaggle input dataset, or the local build folder."""
    cands = [os.environ.get("SIMPT_DATA", "")]
    cands += [str(p.parent) for p in Path("/kaggle/input").rglob("bundle_manifest.json")] if Path("/kaggle/input").exists() else []
    cands += ["../output/simpt_power_release/open_bundle_r2", "output/simpt_power_release/open_bundle_r2"]
    for c in cands:
        if c and (Path(c) / "bundle_manifest.json").exists():
            return Path(c)
    raise FileNotFoundError("SimPT-Power bundle not found; set SIMPT_DATA")
DATA = find_data()
manifest = json.loads((DATA / "bundle_manifest.json").read_text())
print(DATA, "|", manifest["bundle_of"])

In [ ]:
import pandas as pd
pd.DataFrame(manifest["files"]).T[["rows", "description"]]

## HV core: network map by voltage level

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
lines = pd.read_parquet(DATA / "hv/lines.parquet")
geom = lines.geometry_json.map(json.loads)
colors = {400: "#6b3fa0", 220: "#b07cc6", 150: "#e0a3c8", 130: "#9ec5e8", 60: "#9ec5e8"}
fig, ax = plt.subplots(figsize=(5, 8))
for kv in [60, 130, 150, 220, 400]:
    sel = lines.voltage_kv == kv
    ax.add_collection(LineCollection([g for g in geom[sel]], colors=colors[kv], lw=0.4 if kv < 150 else 1.0, label=f"{kv} kV"))
ax.autoscale(); ax.set_aspect(1.3); ax.axis("off"); ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"CORE-3787-REN: {len(lines):,} lines")
plt.show()
lines.groupby("voltage_kv").agg(lines=("line_id", "count"), km=("length_km", "sum")).round(0)

## HV core: 31,492 solved snapshots
One row per 15-minute snapshot, with observed national load and model results. Overloads are kept as computed.

In [ ]:
snap = pd.read_parquet(DATA / "hv/snapshots_summary.parquet")
snap["timestamp_utc"] = pd.to_datetime(snap.timestamp_utc, utc=True)
daily = snap.set_index("timestamp_utc").resample("D").agg({"observed_load_mw": "max", "maximum_line_loading_percent": "max"})
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
daily.observed_load_mw.plot(ax=ax[0]); ax[0].set_ylabel("daily peak load (MW)")
daily.maximum_line_loading_percent.plot(ax=ax[1], color="tab:red"); ax[1].axhline(100, ls="--", c="k", lw=0.8)
ax[1].set_ylabel("max line loading (%)"); plt.tight_layout(); plt.show()
print(f"snapshots {len(snap):,}; converged {snap.converged.sum():,}; with at least one line >100%: {(snap.lines_over_100_percent > 0).sum():,}")

## Secondary substations (PTD) and the geographic LV network

In [ ]:
ptd = pd.read_parquet(DATA / "lv/ptd_public_attributes.parquet")
lvr = pd.read_parquet(DATA / "lv/lv_geo_ptd_results.parquet")
print(f"PTDs: {len(ptd):,}; with a geographic LV network: {len(lvr):,}")
by = lvr.merge(ptd[["ptd_code", "district_name"]], on="ptd_code").groupby("district_name")[["overhead_km", "street_km", "drops_km"]].sum()
by = by.rename(columns={"overhead_km": "pole network (overhead)", "street_km": "street-routed extensions", "drops_km": "service drops"})
by.sort_values("pole network (overhead)").plot.barh(stacked=True, figsize=(7, 6)); plt.xlabel("km"); plt.ylabel("")
plt.title("LV-GEO-A lengths by district"); plt.show()

### One LV feeder
Segments of one PTD: overhead lines on public poles, street-routed extensions and service drops.

In [ ]:
import duckdb
ptd_code = "0911D2009300"
seg = duckdb.sql(f"select * from read_parquet('{DATA}/lv/lv_geo_segments.parquet') where ptd_code = '{ptd_code}'").df()
style = {"overhead": ("tab:blue", 1.4), "underground": ("tab:red", 1.4), "service_drop": ("grey", 0.6)}
fig, ax = plt.subplots(figsize=(6, 6))
for cls, (c, w) in style.items():
    d = seg[seg.cable_class == cls]
    ax.add_collection(LineCollection([[(a, b), (e, f)] for a, b, e, f in zip(d.lon_from, d.lat_from, d.lon_to, d.lat_to)], colors=c, lw=w, label=cls))
ax.autoscale(); ax.set_aspect(1.3); ax.legend(); ax.set_title(ptd_code); plt.show()
seg.groupby("cable_class").length_km.sum().round(2)

## Public observations
Reference tables kept next to the model, e.g. ERSE 2016 LV line lengths by NUTS III region.

In [ ]:
pd.read_parquet(DATA / "observations/erse_bt_nuts3_2016.parquet").head()

**Citation.** SimPT-Power is a simulation database built from public data. It is *not* the operator's network or a digital twin: converged power flows show numerical consistency, not agreement with operator state estimates. Please cite the archived release (Zenodo DOI) and the data paper. Data licences follow the sources (E-REDES CC BY 4.0, OpenStreetMap-derived layers ODbL); see `SOURCE_LICENSES.md` in the release.